# Teacher A — Whisper → IndicTrans2

## Copy/run order

1. Block 1 — configuration
2. Block 2 — Phase 0A runtime probe
3. Block 3 — Phase 0B bundle and audio verification
4. Block 4 — stop gate
5. Block 5 — dependencies (after approval)
6. Block 6 — shared utilities
7. Block 7 — immutable model revisions
8. Block 8 — Whisper ASR
9. Block 9 — IndicTrans2 translation
10. Block 10 — Hindi-2K tokenizer audit
11. Block 11 — metadata
12. Block 12 — Teacher A metrics, diagnostics, and manual examples
13. Block 13 — persistent artifact archive

This notebook is independently runnable in a clean Colab GPU session. It benchmarks English speech → English transcript → Hindi text. Outputs are written incrementally to Google Drive so they survive Colab termination. It never trains Nadantara, changes the source manifest, or overwrites human Hindi references.

Run cells top-to-bottom. The first cells are a hard Phase 0 gate: runtime and uploaded-bundle access only. Do not run the dependency/model cells until that gate passes and the experiment is approved.

Official APIs: [Whisper](https://huggingface.co/openai/whisper-large-v3-turbo), [IndicTrans2](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B). IndicTrans2 requires accepted Hugging Face access conditions and `trust_remote_code=True`; its revision is resolved and recorded before loading.

In [ ]:
# BLOCK 1 — configuration and benchmark bundle paths.
from __future__ import annotations

import gc
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import statistics
import tarfile
import time
import unicodedata
import zlib

import numpy as np
import torch

MODEL_IDS = {
    'speech_teacher': 'openai/whisper-large-v3-turbo',
    'translation_teacher': 'ai4bharat/indictrans2-en-indic-1B',
}
from google.colab import drive
drive.mount('/content/drive')
PERSIST_ROOT = Path('/content/drive/MyDrive/Nadantara/teacher_a_100')
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_ARCHIVE = Path('/content/wordproject_validation_100_seed_20260918.tar.gz')
BUNDLE_ROOT = Path('/content/wordproject_validation_100_seed_20260918')
OUTPUT_ROOT = PERSIST_ROOT / 'outputs'
EXPECTED_BENCHMARK_EXAMPLES = 100
PERSIST_BENCHMARK_ROOT = PERSIST_ROOT / 'benchmark'
RETRY_FAILED = False
MAX_AUDIO_SECONDS = 30.0
MAX_TRANSLATION_TOKENS = 256
print('python=', platform.python_version())
print('bundle_archive=', BUNDLE_ARCHIVE)
print('bundle_root=', BUNDLE_ROOT)
print('output_root=', OUTPUT_ROOT)
print('persistent_root=', PERSIST_ROOT)

In [ ]:
# BLOCK 2 — PHASE 0A runtime probe; this cell downloads no models.
import shutil as _shutil

print('python=', platform.python_version())
print('torch=', torch.__version__)
print('torch_cuda_version=', torch.version.cuda)
print('cuda_available=', torch.cuda.is_available())
print('content_exists=', Path('/content').is_dir())
print('free_disk_gib=', round(_shutil.disk_usage('/content').free / 1024**3, 3))
if not torch.cuda.is_available():
    raise RuntimeError('PHASE 0 STOP: no CUDA GPU is available')
device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
print('gpu_index=', device_index)
print('gpu_name=', props.name)
print('gpu_vram_gib=', round(props.total_memory / 1024**3, 3))
print('gpu_free_memory_gib=', round(free_bytes / 1024**3, 3))
print('gpu_total_memory_gib=', round(total_bytes / 1024**3, 3))
print('PHASE_0_RUNTIME_PASS')

In [ ]:
# BLOCK 3 — PHASE 0B uploaded-file access, manifest identity, and audio-byte checks.
if not BUNDLE_ARCHIVE.is_file():
    raise FileNotFoundError(f'upload the benchmark archive first: {BUNDLE_ARCHIVE}')
def _safe_extract(archive: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        root = destination.parent.resolve()
        for member in tar.getmembers():
            target = (destination.parent / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        tar.extractall(destination.parent)
if not (BUNDLE_ROOT / 'benchmark_manifest.jsonl').is_file():
    _safe_extract(BUNDLE_ARCHIVE, BUNDLE_ROOT)
manifest_path = BUNDLE_ROOT / 'benchmark_manifest.jsonl'
metadata_path = BUNDLE_ROOT / 'bundle_metadata.json'
sample_ids_path = BUNDLE_ROOT / 'sample_ids.txt'
rows = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
if not rows:
    raise RuntimeError('benchmark manifest is empty')
if len(rows) != EXPECTED_BENCHMARK_EXAMPLES:
    raise RuntimeError(f'expected exactly {EXPECTED_BENCHMARK_EXAMPLES} benchmark rows, got {len(rows)}')
if len({row['sample_id'] for row in rows}) != len(rows):
    raise RuntimeError('duplicate sample IDs in benchmark manifest')
def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()
bundle_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
if not sample_ids_path.is_file():
    raise FileNotFoundError(sample_ids_path)
sample_ids_text = sample_ids_path.read_text(encoding='utf-8')
if hashlib.sha256(sample_ids_text.encode('utf-8')).hexdigest() != bundle_metadata['sample_ids_sha256']:
    raise RuntimeError('sample_ids.txt hash does not match bundle metadata')
if _sha256(manifest_path) != bundle_metadata['manifest_sha256']:
    raise RuntimeError('benchmark manifest hash does not match bundle metadata')
for row in rows:
    audio_path = BUNDLE_ROOT / row['audio_path']
    if not audio_path.is_file():
        raise FileNotFoundError(audio_path)
    actual_hash = _sha256(audio_path)
    if actual_hash != row['audio_sha256']:
        raise RuntimeError(f'audio hash mismatch for {row["sample_id"]}')
print('samples=', len(rows))
print('sample_ids=', [row['sample_id'] for row in rows])
PERSIST_BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
for source_path in (BUNDLE_ARCHIVE, manifest_path, sample_ids_path, metadata_path):
    shutil.copy2(source_path, PERSIST_BENCHMARK_ROOT / source_path.name)
print('bundle_metadata=', bundle_metadata)
print('persistent_benchmark_root=', PERSIST_BENCHMARK_ROOT)
print('PHASE_0_FILES_PASS')

## BLOCK 4 — Phase 0 stop gate

At this point confirm `PHASE_0_RUNTIME_PASS` and `PHASE_0_FILES_PASS`, and inspect the printed GPU, VRAM, sample IDs, and hashes. Stop here unless the 100-example Teacher A run is explicitly approved. The next cell installs teacher-specific dependencies; it does not run a model by itself.

In [ ]:
# BLOCK 5 — POST-APPROVAL ONLY — Teacher A's isolated dependencies.
%pip install -q 'transformers==4.53.2' accelerate sentencepiece IndicTransToolkit soundfile scipy psutil huggingface_hub sacrebleu

In [ ]:
# BLOCK 6 — shared Teacher A utilities.
import gc
import math
import psutil
import soundfile as sf
from scipy.signal import resample_poly

TARGET_RATE = 16_000
def _token() -> str | None:
    value = os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata
        value = value or userdata.get('HF_TOKEN')
    except Exception:
        pass
    return value or None
HF_TOKEN = _token()
def normalize_hindi(text: str) -> str:
    return ' '.join(unicodedata.normalize('NFC', text).split())
def audio_sha256(path: Path) -> str:
    return _sha256(path)
def read_audio(path: Path) -> tuple[np.ndarray, float, int]:
    array, sample_rate = sf.read(path, dtype='float32', always_2d=True)
    waveform = array.mean(axis=1) if array.shape[1] > 1 else array[:, 0]
    if sample_rate != TARGET_RATE:
        divisor = math.gcd(int(sample_rate), TARGET_RATE)
        waveform = resample_poly(waveform, TARGET_RATE // divisor, int(sample_rate) // divisor).astype('float32')
        sample_rate = TARGET_RATE
    waveform = np.clip(waveform, -1.0, 1.0).astype('float32')
    duration = len(waveform) / TARGET_RATE
    if duration > MAX_AUDIO_SECONDS:
        raise ValueError(f'audio exceeds {MAX_AUDIO_SECONDS}s: {duration:.3f}s')
    return waveform, duration, int(sample_rate)
def existing_records(path: Path) -> dict[str, dict[str, object]]:
    if not path.is_file(): return {}
    records = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row = json.loads(line); records[str(row['sample_id'])] = row
    return records
def append_jsonl(path: Path, row: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
        stream.flush()
def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print('transformers=', __import__('transformers').__version__)
print('HF_TOKEN_available=', bool(HF_TOKEN))
durations = []
for row in rows:
    try:
        _, duration, _ = read_audio(BUNDLE_ROOT / row['audio_path'])
        durations.append(duration)
    except Exception as exc:
        print('duration_error=', row['sample_id'], repr(exc))
if durations:
    print('duration_min_median_p90_p95_max=', min(durations), statistics.median(durations), np.percentile(durations, 90), np.percentile(durations, 95), max(durations))

In [ ]:
# BLOCK 7 — POST-APPROVAL ONLY — verify the immutable smoke-tested model revisions.
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
MODEL_REVISIONS = {
    'openai/whisper-large-v3-turbo': '41f01f3fe87f28c78e2fbf8b568835947dd65ed9',
    'ai4bharat/indictrans2-en-indic-1B': '10e65a9951a1e922cd109a95e8aba9357b62144b',
}
for model_id, revision in MODEL_REVISIONS.items():
    resolved = api.model_info(model_id, revision=revision, token=HF_TOKEN).sha
    if resolved != revision:
        raise RuntimeError(f'model revision verification failed for {model_id}: expected {revision}, got {resolved}')
print(json.dumps(MODEL_REVISIONS, indent=2))

In [ ]:
# BLOCK 8 — POST-APPROVAL ONLY — Whisper ASR stage. One sample per checkpointed JSONL record.
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

asr_path = OUTPUT_ROOT / 'teacher_a_whisper_transcripts.jsonl'
asr_records = existing_records(asr_path)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
whisper_processor = AutoProcessor.from_pretrained(MODEL_IDS['speech_teacher'], revision=MODEL_REVISIONS[MODEL_IDS['speech_teacher']], token=HF_TOKEN)
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_IDS['speech_teacher'], revision=MODEL_REVISIONS[MODEL_IDS['speech_teacher']], torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True, token=HF_TOKEN).to(device)
whisper_model.eval()
whisper_generation_config = {'language': 'english', 'task': 'transcribe', 'do_sample': False, 'num_beams': 1, 'max_new_tokens': 256}
for row in rows:
    sample_id = row['sample_id']
    if sample_id in asr_records and (asr_records[sample_id].get('status') == 'ok' or not RETRY_FAILED): continue
    base = {'schema_version': 1, 'sample_id': sample_id, 'audio_path': row['audio_path'], 'audio_sha256': row['audio_sha256'], 'source_manifest_sha256': row.get('source_manifest_sha256'), 'source_dataset': row.get('source_dataset'), 'source_dataset_revision': row.get('source_dataset_revision'), 'source_license': row.get('source_license'), 'source_url': row.get('source_url'), 'source_split': row.get('source_split'), 'reference_hindi': row.get('reference_hindi'), 'source_text': row.get('source_text'), 'source_asr_text': row.get('source_asr_text'), 'speech_teacher': MODEL_IDS['speech_teacher'], 'speech_teacher_revision': MODEL_REVISIONS[MODEL_IDS['speech_teacher']], 'speech_teacher_license': 'MIT', 'status': 'error', 'stage': 'whisper'}
    started = time.perf_counter()
    try:
        waveform, duration, _ = read_audio(BUNDLE_ROOT / row['audio_path'])
        inputs = whisper_processor(waveform, sampling_rate=TARGET_RATE, return_tensors='pt', return_attention_mask=True)
        inputs = {key: (value.to(device=device, dtype=dtype) if torch.is_floating_point(value) else value.to(device)) for key, value in inputs.items()}
        forced_ids = whisper_processor.get_decoder_prompt_ids(language='english', task='transcribe')
        with torch.inference_mode():
            generated = whisper_model.generate(**inputs, forced_decoder_ids=forced_ids, do_sample=False, num_beams=1, max_new_tokens=256, return_dict_in_generate=True, output_scores=True)
        transcript = whisper_processor.batch_decode(generated.sequences, skip_special_tokens=True)[0].strip()
        mean_logprob = None
        if generated.scores:
            transition = whisper_model.compute_transition_scores(generated.sequences, generated.scores, normalize_logits=True)
            finite = transition[0][torch.isfinite(transition[0])]
            mean_logprob = float(finite.mean().cpu()) if finite.numel() else None
        base.update({'status': 'ok', 'stage': None, 'whisper_transcript_en': transcript, 'asr_seconds': time.perf_counter() - started, 'duration_seconds': duration, 'whisper_mean_logprob': mean_logprob, 'whisper_generation_config': whisper_generation_config})
    except Exception as exc:
        base.update({'error': repr(exc), 'asr_seconds': time.perf_counter() - started})
    append_jsonl(asr_path, base); asr_records[sample_id] = base
    print(sample_id, base['status'], base.get('error', ''))
del whisper_model, whisper_processor
clear_cuda()
print('asr_output=', asr_path)
print('asr_records=', len(asr_records), 'successful=', sum(row.get('status') == 'ok' for row in asr_records.values()), 'failed=', sum(row.get('status') != 'ok' for row in asr_records.values()))

In [ ]:
# BLOCK 9 — POST-APPROVAL ONLY — IndicTrans2 text translation stage.
from IndicTransToolkit.processor import IndicProcessor
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_path = OUTPUT_ROOT / 'teacher_whisper_indictrans2_100.jsonl'
teacher_records = existing_records(teacher_path)
successful_asr = [row for row in rows if asr_records.get(row['sample_id'], {}).get('status') == 'ok']
if not successful_asr:
    raise RuntimeError('No successful Whisper ASR records. Rerun the ASR cell with RETRY_FAILED=True before starting IndicTrans2.')
print('successful_asr=', len(successful_asr), 'requested=', len(rows))
translation_id = MODEL_IDS['translation_teacher']
translation_revision = MODEL_REVISIONS[translation_id]
translation_tokenizer = AutoTokenizer.from_pretrained(translation_id, revision=translation_revision, trust_remote_code=True, token=HF_TOKEN)
translation_device = 'cuda' if torch.cuda.is_available() else 'cpu'
translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_id, revision=translation_revision, trust_remote_code=True, attn_implementation='eager', low_cpu_mem_usage=True, token=HF_TOKEN).to(translation_device)
if translation_device == 'cuda': translation_model.half()
translation_model.eval()
indic_processor = IndicProcessor(inference=True)
translation_generation_config = {'src_lang': 'eng_Latn', 'tgt_lang': 'hin_Deva', 'do_sample': False, 'num_beams': 5, 'num_return_sequences': 1, 'max_length': MAX_TRANSLATION_TOKENS, 'batch_size': 1}
for row in rows:
    sample_id = row['sample_id']; asr = asr_records.get(sample_id, {})
    if sample_id in teacher_records and (teacher_records[sample_id].get('status') == 'ok' or not RETRY_FAILED): continue
    base = {'schema_version': 1, 'sample_id': sample_id, 'audio_path': row['audio_path'], 'audio_sha256': row['audio_sha256'], 'source_manifest_sha256': row.get('source_manifest_sha256'), 'source_dataset': row.get('source_dataset'), 'source_dataset_revision': row.get('source_dataset_revision'), 'source_license': row.get('source_license'), 'source_url': row.get('source_url'), 'source_split': row.get('source_split'), 'reference_hindi': row.get('reference_hindi'), 'source_text': row.get('source_text'), 'source_asr_text': row.get('source_asr_text'), 'whisper_transcript_en': asr.get('whisper_transcript_en'), 'speech_teacher': MODEL_IDS['speech_teacher'], 'speech_teacher_revision': MODEL_REVISIONS[MODEL_IDS['speech_teacher']], 'speech_teacher_license': 'MIT', 'translation_teacher': translation_id, 'translation_teacher_revision': translation_revision, 'translation_teacher_license': 'MIT', 'teacher_pipeline': 'whisper-large-v3-turbo+indictrans2-en-indic-1B', 'status': 'error', 'stage': 'indictrans2', 'generation_config': translation_generation_config}
    started = time.perf_counter()
    if asr.get('status') != 'ok':
        base.update({'stage': 'whisper', 'error': asr.get('error', 'missing ASR output'), 'asr_seconds': asr.get('asr_seconds')})
    else:
        try:
            batch = indic_processor.preprocess_batch([str(asr['whisper_transcript_en'])], src_lang='eng_Latn', tgt_lang='hin_Deva')
            inputs = translation_tokenizer(batch, truncation=True, padding='longest', return_tensors='pt', return_attention_mask=True).to(translation_device)
            with torch.inference_mode():
                generated = translation_model.generate(**inputs, use_cache=True, min_length=0, max_length=MAX_TRANSLATION_TOKENS, num_beams=5, num_return_sequences=1)
            decoded = translation_tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            postprocessed = indic_processor.postprocess_batch(decoded, lang='hin_Deva')[0]
            base.update({'status': 'ok', 'stage': None, 'teacher_hindi_model_decoded_raw': decoded[0], 'teacher_hindi_raw': postprocessed, 'teacher_hindi_normalized': normalize_hindi(postprocessed), 'asr_seconds': asr.get('asr_seconds'), 'translation_seconds': time.perf_counter() - started})
        except Exception as exc:
            base.update({'error': repr(exc), 'translation_seconds': time.perf_counter() - started, 'asr_seconds': asr.get('asr_seconds')})
    append_jsonl(teacher_path, base); teacher_records[sample_id] = base
    print(sample_id, base['status'], base.get('stage', 'ok'), base.get('error', ''))
del translation_model, translation_tokenizer, indic_processor
clear_cuda()
print('canonical_output=', teacher_path)
print('translation_records=', len(teacher_records), 'successful=', sum(row.get('status') == 'ok' for row in teacher_records.values()), 'failed=', sum(row.get('status') != 'ok' for row in teacher_records.values()))

In [ ]:
# BLOCK 10 — Hindi-2K tokenizer audit. Upload the exact baseline tokenizer first.
import sentencepiece as spm
TOKENIZER_PATH = Path('/content/hindi_2k.model')
EXPECTED_TOKENIZER_SHA256 = '34b1c9c8f24b9fdb27a393978a0034fd96ea35b835b1a667e4373098a483137d'
MAX_SEMANTIC_TOKENS = 128
if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f'upload the exact baseline tokenizer first: {TOKENIZER_PATH}')
tokenizer_sha256 = _sha256(TOKENIZER_PATH)
if tokenizer_sha256 != EXPECTED_TOKENIZER_SHA256:
    raise RuntimeError(f'tokenizer SHA-256 mismatch: expected {EXPECTED_TOKENIZER_SHA256}, got {tokenizer_sha256}')
tokenizer = spm.SentencePieceProcessor(model_file=str(TOKENIZER_PATH))
output_rows = [teacher_records[row['sample_id']] for row in rows if row['sample_id'] in teacher_records]
successful_outputs = [row for row in output_rows if row.get('status') == 'ok']
lengths = []; tokens_per_word = []; tokens_per_character = []; unknown = 0; total = 0; fragmentation_examples = []
for row in successful_outputs:
    text = str(row.get('teacher_hindi_raw', ''))
    ids = tokenizer.encode(text, out_type=int)
    pieces = tokenizer.encode(text, out_type=str)
    word_count = len(text.split())
    character_count = sum(not character.isspace() for character in text)
    lengths.append(len(ids)); tokens_per_word.append(len(ids) / max(1, word_count)); tokens_per_character.append(len(ids) / max(1, character_count))
    unknown += sum(token == tokenizer.unk_id() for token in ids); total += len(ids)
    if len(fragmentation_examples) < 20 and (any(character.isascii() and character.isalpha() for character in text) or any(character.isdigit() for character in text) or any(not character.isalnum() and not character.isspace() for character in text)):
        fragmentation_examples.append({'sample_id': row['sample_id'], 'teacher_hindi_raw': text, 'pieces': pieces, 'token_count': len(ids)})
def percentile(values, value):
    return float(np.percentile(values, value)) if values else None
tokenizer_audit = {
    'schema_version': 1,
    'tokenizer_path': 'artifacts/tokenizers/hindi_2k/hindi.model',
    'tokenizer_sha256': tokenizer_sha256,
    'vocab_size': tokenizer.vocab_size(),
    'max_semantic_tokens': MAX_SEMANTIC_TOKENS,
    'successful_outputs': len(successful_outputs),
    'unknown_token_count': unknown,
    'unknown_token_rate': unknown / max(1, total),
    'token_length_p50': percentile(lengths, 50),
    'token_length_p90': percentile(lengths, 90),
    'token_length_p95': percentile(lengths, 95),
    'token_length_p99': percentile(lengths, 99),
    'token_length_max': max(lengths) if lengths else None,
    'mean_tokens_per_hindi_word': float(np.mean(tokens_per_word)) if tokens_per_word else None,
    'mean_tokens_per_unicode_character': float(np.mean(tokens_per_character)) if tokens_per_character else None,
    'percentage_exceeding_max_semantic_tokens': 100.0 * sum(length + 1 > MAX_SEMANTIC_TOKENS for length in lengths) / max(1, len(lengths)),
    'fragmentation_examples': fragmentation_examples,
    'lengths_include_special_tokens': False,
    'exceedance_rule': 'raw SentencePiece pieces + BOS must fit within max_semantic_tokens',
}
(OUTPUT_ROOT / 'teacher_a_tokenizer_audit.json').write_text(json.dumps(tokenizer_audit, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(tokenizer_audit, ensure_ascii=False, indent=2, sort_keys=True))

In [ ]:
# BLOCK 11 — persist reproducibility metadata without secrets.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
metadata = {
    'schema_version': 1,
    'teacher_pipeline': 'whisper-large-v3-turbo+indictrans2-en-indic-1B',
    'model_ids': MODEL_IDS,
    'model_revisions': globals().get('MODEL_REVISIONS', {}),
    'model_licenses': {'openai/whisper-large-v3-turbo': 'MIT', 'ai4bharat/indictrans2-en-indic-1B': 'MIT'},
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'transformers': __import__('transformers').__version__,
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'precision': str(dtype),
    'bundle_metadata': json.loads((BUNDLE_ROOT / 'bundle_metadata.json').read_text(encoding='utf-8')),
    'generation_config': {'whisper': globals().get('whisper_generation_config', {}), 'indictrans2': globals().get('translation_generation_config', {})},
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'benchmark_expected_examples': EXPECTED_BENCHMARK_EXAMPLES,
    'persistent_root': str(PERSIST_ROOT),
    'artifact_paths': {'whisper_transcripts': str(OUTPUT_ROOT / 'teacher_a_whisper_transcripts.jsonl'), 'teacher_a_jsonl': str(OUTPUT_ROOT / 'teacher_whisper_indictrans2_100.jsonl'), 'tokenizer_audit': str(OUTPUT_ROOT / 'teacher_a_tokenizer_audit.json')},
}
(OUTPUT_ROOT / 'teacher_a_experiment_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(metadata, indent=2, ensure_ascii=False, sort_keys=True))
print('teacher_a_metadata=', OUTPUT_ROOT / 'teacher_a_experiment_metadata.json')

In [ ]:
# BLOCK 12 — Teacher A metrics, ASR diagnostics, failures, and manual examples.
import re
import sacrebleu

def _norm(text):
    return ' '.join(unicodedata.normalize('NFC', str(text or '')).split())
def _diagnostics(text, source):
    text = _norm(text); source = _norm(source)
    words = text.split(); source_words = source.split()
    devanagari = sum('\u0900' <= char <= '\u097f' for char in text)
    letters = sum(char.isalpha() for char in text)
    latin_ratio = sum('a' <= char.lower() <= 'z' for char in text) / max(1, letters)
    repeated = sum(a == b for a, b in zip(words, words[1:]))
    return {'empty': not text, 'mostly_latin': bool(letters) and latin_ratio > 0.8, 'devanagari_ratio': devanagari / max(1, len(text)), 'adjacent_word_repetition_rate': repeated / max(1, len(words) - 1), 'source_copy': bool(text) and text.casefold() == source.casefold(), 'word_count': len(words), 'char_count': len(text), 'source_word_count': len(source_words), 'target_source_word_ratio': len(words) / max(1, len(source_words))}
def _edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for index, ref_item in enumerate(reference, start=1):
        current = [index]
        for hyp_index, hyp_item in enumerate(hypothesis, start=1):
            current.append(min(current[-1] + 1, previous[hyp_index] + 1, previous[hyp_index - 1] + (ref_item != hyp_item)))
        previous = current
    return previous[-1]
def _wer(reference, hypothesis):
    reference_words = _norm(reference).casefold().split(); hypothesis_words = _norm(hypothesis).casefold().split()
    return _edit_distance(reference_words, hypothesis_words) / max(1, len(reference_words))
def _cer(reference, hypothesis):
    reference_chars = list(_norm(reference).casefold()); hypothesis_chars = list(_norm(hypothesis).casefold())
    return _edit_distance(reference_chars, hypothesis_chars) / max(1, len(reference_chars))
teacher_rows = [teacher_records.get(row['sample_id'], {'sample_id': row['sample_id'], 'status': 'missing'}) for row in rows]
successful_teacher = [row for row in teacher_rows if row.get('status') == 'ok']
hypotheses = [_norm(row.get('teacher_hindi_raw', '')) for row in successful_teacher]
references = [_norm(row.get('reference_hindi', '')) for row in successful_teacher]
diagnostic_rows = [_diagnostics(row.get('teacher_hindi_raw', ''), row.get('source_text', '')) for row in successful_teacher]
asr_successful = [asr_records.get(row['sample_id']) for row in rows if asr_records.get(row['sample_id'], {}).get('status') == 'ok']
reference_english_field = 'source_asr_text' if any(row.get('source_asr_text') for row in rows) else 'source_text'
wer_values = [_wer(row.get(reference_english_field, ''), asr_records[row['sample_id']].get('whisper_transcript_en', '')) for row in rows if row['sample_id'] in asr_records and asr_records[row['sample_id']].get('status') == 'ok']
cer_values = [_cer(row.get(reference_english_field, ''), asr_records[row['sample_id']].get('whisper_transcript_en', '')) for row in rows if row['sample_id'] in asr_records and asr_records[row['sample_id']].get('status') == 'ok']
source_target_ratios = [float(item['target_source_word_ratio']) for item in diagnostic_rows]
teacher_a_metrics = {
    'schema_version': 1,
    'requested_samples': len(rows),
    'successful_asr_samples': len(asr_successful),
    'successful_translation_samples': len(successful_teacher),
    'failed_asr_samples': len(rows) - len(asr_successful),
    'failed_translation_samples': len(rows) - len(successful_teacher),
    'bleu': float(sacrebleu.corpus_bleu(hypotheses, [references], tokenize='intl', use_effective_order=True).score) if successful_teacher else None,
    'chrf2': float(sacrebleu.corpus_chrf(hypotheses, [references], word_order=2).score) if successful_teacher else None,
    'exact_match_count': sum(hypothesis == reference for hypothesis, reference in zip(hypotheses, references, strict=True)),
    'empty_output_count': sum(bool(item['empty']) for item in diagnostic_rows),
    'mostly_latin_count': sum(bool(item['mostly_latin']) for item in diagnostic_rows),
    'source_copy_count': sum(bool(item['source_copy']) for item in diagnostic_rows),
    'mean_devanagari_ratio': float(np.mean([item['devanagari_ratio'] for item in diagnostic_rows])) if diagnostic_rows else None,
    'mean_adjacent_word_repetition_rate': float(np.mean([item['adjacent_word_repetition_rate'] for item in diagnostic_rows])) if diagnostic_rows else None,
    'target_word_count_p50_p90_p95_p99_max': [float(np.percentile([item['word_count'] for item in diagnostic_rows], value)) for value in (50, 90, 95, 99)] + ([max(item['word_count'] for item in diagnostic_rows)] if diagnostic_rows else [None]),
    'target_source_word_ratio_p50_p90_p95_p99_max': [float(np.percentile(source_target_ratios, value)) for value in (50, 90, 95, 99)] + ([max(source_target_ratios)] if source_target_ratios else [None]),
    'asr_reference_field': reference_english_field,
    'asr_wer_mean': float(np.mean(wer_values)) if wer_values else None,
    'asr_cer_mean': float(np.mean(cer_values)) if cer_values else None,
    'asr_total_seconds': sum(float(row.get('asr_seconds', 0.0) or 0.0) for row in asr_records.values()),
    'translation_total_seconds': sum(float(row.get('translation_seconds', 0.0) or 0.0) for row in teacher_records.values()),
}
failure_rows = [row for row in teacher_rows if row.get('status') != 'ok'] + [row for row in asr_records.values() if row.get('status') != 'ok']
with (OUTPUT_ROOT / 'teacher_a_failures.jsonl').open('w', encoding='utf-8') as stream:
    for row in failure_rows:
        stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
ranked = []
for row, diagnostic in zip(successful_teacher, diagnostic_rows, strict=True):
    score = 0
    score += 5 if diagnostic['empty'] else 0; score += 4 if diagnostic['mostly_latin'] else 0; score += 3 if diagnostic['source_copy'] else 0; score += 3 if diagnostic['adjacent_word_repetition_rate'] > 0 else 0; score += 2 if any(character.isdigit() for character in str(row.get('teacher_hindi_raw', ''))) else 0
    ranked.append((score, row['sample_id'], row, diagnostic))
benchmark_by_id = {row['sample_id']: row for row in rows}
manual_ids = []
for row in rows:
    sample_id = row['sample_id']
    if asr_records.get(sample_id, {}).get('status') != 'ok' or teacher_records.get(sample_id, {}).get('status') != 'ok':
        manual_ids.append(sample_id)
for _, sample_id, _, _ in sorted(ranked, key=lambda item: (-item[0], item[1])):
    if sample_id not in manual_ids:
        manual_ids.append(sample_id)
    if len(manual_ids) >= 20:
        break
manual_ids = manual_ids[:20]
manual_examples = []
for sample_id in manual_ids:
    bundle_row = benchmark_by_id[sample_id]; teacher_row = teacher_records.get(sample_id, {}); asr_row = asr_records.get(sample_id, {})
    manual_examples.append({'sample_id': sample_id, 'reference_english': bundle_row.get(reference_english_field), 'whisper_transcript_en': asr_row.get('whisper_transcript_en'), 'human_hindi': bundle_row.get('reference_hindi'), 'teacher_a_hindi': teacher_row.get('teacher_hindi_raw'), 'status': teacher_row.get('status', 'missing'), 'error': teacher_row.get('error') or asr_row.get('error')})
(OUTPUT_ROOT / 'teacher_a_metrics.json').write_text(json.dumps(teacher_a_metrics, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'teacher_a_manual_examples.json').write_text(json.dumps(manual_examples, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
with (OUTPUT_ROOT / 'teacher_a_manual_examples.md').open('w', encoding='utf-8') as stream:
    for example in manual_examples:
        stream.write(f"## {example['sample_id']}\n\nReference English: {example.get('reference_english', '')}\n\nWhisper: {example.get('whisper_transcript_en', '')}\n\nHuman Hindi: {example.get('human_hindi', '')}\n\nTeacher A Hindi: {example.get('teacher_a_hindi', '')}\n\nStatus: {example.get('status')}\n\n---\n\n")
print(json.dumps(teacher_a_metrics, ensure_ascii=False, indent=2, sort_keys=True))
print('teacher_a_manual_examples=', OUTPUT_ROOT / 'teacher_a_manual_examples.md')

In [ ]:
# BLOCK 13 — verify and package persistent Teacher A artifacts.
import zipfile
required_artifacts = [
    PERSIST_BENCHMARK_ROOT / 'benchmark_manifest.jsonl',
    PERSIST_BENCHMARK_ROOT / 'sample_ids.txt',
    PERSIST_BENCHMARK_ROOT / 'bundle_metadata.json',
    PERSIST_BENCHMARK_ROOT / BUNDLE_ARCHIVE.name,
    OUTPUT_ROOT / 'teacher_a_whisper_transcripts.jsonl',
    OUTPUT_ROOT / 'teacher_whisper_indictrans2_100.jsonl',
    OUTPUT_ROOT / 'teacher_a_experiment_metadata.json',
    OUTPUT_ROOT / 'teacher_a_metrics.json',
    OUTPUT_ROOT / 'teacher_a_tokenizer_audit.json',
    OUTPUT_ROOT / 'teacher_a_manual_examples.json',
    OUTPUT_ROOT / 'teacher_a_manual_examples.md',
    OUTPUT_ROOT / 'teacher_a_failures.jsonl',
 ]
missing = [str(path) for path in required_artifacts if not path.is_file()]
if missing:
    raise FileNotFoundError('missing persistent artifacts: ' + ', '.join(missing))
archive_path = PERSIST_ROOT / 'teacher_a_100_artifacts.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in required_artifacts:
        archive.write(path, arcname=path.relative_to(PERSIST_ROOT))
print('teacher_a_archive=', archive_path)
print('teacher_a_archive_bytes=', archive_path.stat().st_size)
print('TEACHER_A_STOP_GATE_READY')